# A1+A2: свип данных × моделей (ROADMAP A; гипотеза: CRL-Prompt —
# low-resource метод, его вклад виден при малых данных/малой модели)

Дизайн: {MLE, mask, RL} × размеры {250/500/1000/[6000]} × 3 сида,
**без probe** — общий СЛУЧАЙНЫЙ старт на (модель, сид) переиспользуется
всеми конфигами и размерами (парность 2 уровней). Протокол/код — копии
experiment.ipynb (D1-классификация; датасет статьи). 6000-точка для 3B
берётся из готового D1.6 (n=7 сидов) — не перезапускается.

Время: 3B ~5.5 ч + 0.5B ~2.5 ч. Resume-safe (results.json на прогон).
Сетка/hипотезы зафиксированы априори; все ячейки в отчёт (не p-hacking).

Порядок: сверху вниз (инфраструктура = копии ячеек experiment.ipynb,
дальше свип-блок).

In [2]:
import time, os, gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# CUBLAS_WORKSPACE_CONFIG НЕ включаем: окружение новых сидов должно
# совпадать с завершёнными прогонами (смешивать числовые среды между
# сидами таблицы не хотим); детерминизм уже показан эмпирически
# (каузальный тест, §5.5: бит-в-бит воспроизведение)

# ============================================================
# CELL 1: Imports
# ============================================================
import torch
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    get_cosine_schedule_with_warmup,
    set_seed,
)
from peft import PrefixTuningConfig, TaskType, get_peft_model
from datasets import load_dataset

import numpy as np
import evaluate

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {vram_gb:.1f} GB")


<VENV>/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM: 15.5 GB


In [3]:
# ============================================================
# CELL 2: Config — исследование-2, итерация D1.6
# Апгрейд по итогам внешней критики:
#   1) модель 1.5B -> 3B (реально крупнее статьи);
#   2) настоящий батч 8x1 вместо 1x8 (последовательности ~100 токенов);
#   3) ЧИСТЫЙ eval: вход больше НЕ содержит золотую метку (баг D1.x:
#      generate() продолжал ЗОЛОТУЮ метку — «предсказание» было эхом);
#   4) вербализатор-скоринг (протокол статьи: top-scoring output token);
#   5) per-example корректности -> парный бутстреп (стат. мощность);
#   6) жёсткие негативы для wrong_label (путаемые группы эмоций);
#   7) значения статьи β=0.3, τ=0.1 отдельным конфигом.
# ============================================================
class CFG:
    # ---- 1. Core
    seed = 42
    output_dir = "./gen_dialogue_ed_full"

    gradient_checkpointing = False
    use_cache = True
    bf16 = True
    fp16 = False

    # ---- 2. Model / PEFT
    # D1.6: Qwen2.5-3B (~6.2 ГБ bf16) — заметно крупнее Falcon-rw-1b
    # из статьи (критика: 1.5B был тем же классом). Влезает в 16 ГБ
    # с батчем 8 при seq ~110.
    model_name = "Qwen/Qwen2.5-3B"
    trust_remote_code = True
    peft_kind = "prefix_tuning"
    peft_method = "prefix_tuning"
    num_virtual_tokens = 20
    prefix_projection = True

    # Probe = ГЕНЕРАТОР общего старта
    init_selection_inits = 4
    init_selection_probe_steps = 200

    # ---- 3. Task / Dataset
    task_kind = "emotion_cls"
    dataset_repo = "empathetic_dialogues"
    dataset_revision = "refs/convert/parquet"
    dataset_name = None  # легаси XSum удалено (критика-2: техдолг);
                         # ветка не-cls сейчас не используется

    text_column = "text"
    target_column = "label"

    train_size = 6000
    val_size = 200
    test_size = 500

    max_source_len = 96
    max_target_len = 8
    max_total_len = max_source_len + max_target_len

    prompt_template = "Situation: {source}\nEmotion:"
    padding_side = "left"

    eval_max_new_tokens = 6
    gen_max_new_tokens = 6

    if task_kind == "emotion_cls":
        eval_metrics = ["accuracy", "verb_accuracy"]
    else:
        eval_metrics = ["rouge1", "rouge2", "rougeL"]

    # ---- 4. Schedule
    # D1.6: настоящий батч 8 x accum 1 (та же эффективная партия 8,
    # но GPU-утилизация вместо цикла микробатчей — прогоны в разы
    # быстрее; статистика градиента та же, аккумулирование == батчингу).
    # Эпох 4: 3B сходится быстрее, best-эпохи в D1.5 были 1-3.
    phase1_epochs = 4
    total_epochs = 4
    batch_size = 8
    eval_batch_size = 8
    gradient_accumulation_steps = 1

    learning_rate = 5e-4
    phase2_lr = 1e-4
    weight_decay = 0.0
    warmup_steps = 200

    lr_scheduler = "cosine"
    max_grad_norm = 1.0
    logging_steps = 10
    eval_strategy = "epoch"
    save_strategy = "epoch"
    save_total_limit = 1
    load_best_model_at_end = False
    remove_unused_columns = False
    dataloader_pin_memory = True
    report_to = "none"

    # ---- 5. Reward
    gen_do_sample = False
    gen_temperature = 0.7
    gen_top_p = 0.9
    reward_metric = "accuracy"

    # ---- 6. Contrast
    alpha = 1.0
    beta = 0.1
    contrast_mode = "mask"
    contrast_from_start = True
    contrastive_dropout = 0.1
    contrastive_tau = 0.5
    k_negatives = 1

    gamma = 2e-5
    gamma_auto = True
    gamma_target_frac = 0.3
    sigma = 0.02
    rl_interval = 50
    rl_num_batches = 16
    rl_subset_size = max(16, int(train_size * 0.1))
    use_reward_baseline = True

    divergence_ce_threshold = 8.0
    guard_start_step = 100

    # D1.6: отбор лучшей эпохи по ВСЕМ 200 val-примерам (было 64)
    val_rouge_examples = 200


cfg = CFG()

assert cfg.peft_method == "prefix_tuning"
assert cfg.gradient_checkpointing is False

print("Config OK (D1.6)")
print(f"Model: {cfg.model_name}  <-- апгрейд 1.5B -> 3B")
print(f"Task: {cfg.task_kind} | metrics: {cfg.eval_metrics}")
print(f"PEFT: {cfg.peft_kind} (prefix_projection={cfg.prefix_projection})")
print(f"Dataset: {cfg.dataset_repo} @ {cfg.dataset_revision}")
print(f"Train {cfg.train_size} | val {cfg.val_size} (полная, отбор эпохи) | test {cfg.test_size}")
print(f"Batch: {cfg.batch_size} x accum {cfg.gradient_accumulation_steps} (настоящий батч)")
print(f"Epochs: {cfg.total_epochs} (одна фаза, контраст с 1-й эпохи)")
print(f"LR: {cfg.learning_rate}, warmup: {cfg.warmup_steps}")
print(f"Contrast: mode={cfg.contrast_mode}, beta={cfg.beta}, tau={cfg.contrastive_tau}")
print(f"Eval: ЧИСТЫЙ (без золотой метки во входе) + вербализатор")


Config OK (D1.6)
Model: Qwen/Qwen2.5-3B  <-- апгрейд 1.5B -> 3B
Task: emotion_cls | metrics: ['accuracy', 'verb_accuracy']
PEFT: prefix_tuning (prefix_projection=True)
Dataset: empathetic_dialogues @ refs/convert/parquet
Train 6000 | val 200 (полная, отбор эпохи) | test 500
Batch: 8 x accum 1 (настоящий батч)
Epochs: 4 (одна фаза, контраст с 1-й эпохи)
LR: 0.0005, warmup: 200
Contrast: mode=mask, beta=0.1, tau=0.5
Eval: ЧИСТЫЙ (без золотой метки во входе) + вербализатор


In [4]:
# ============================================================
# CELL 3: Data loading & tokenization
# D1.6: жёсткие негативы wrong_label из групп путаемых эмоций
# (критика D1.5: случайная чужая метка слишком легка) + первые
# токены меток для вербализатор-скоринга
# ============================================================
import re
import random
from huggingface_hub import hf_hub_download, list_repo_files
from datasets import Dataset as HFDataset

tok = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
tok.padding_side = cfg.padding_side

if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id

if cfg.task_kind == "emotion_cls":
    def load_ed_episodes(split):
        """ED — script-датасет, новыми datasets не грузится; читаем
        parquet-конвертацию с Hub. Эпизоды = уникальные conv_id."""
        files = list_repo_files(cfg.dataset_repo, repo_type="dataset",
                                revision=cfg.dataset_revision)
        names = sorted(f for f in files
                       if f.startswith(f"default/{split}/") and f.endswith(".parquet"))
        assert names, f"нет parquet-шардов для split={split}"
        paths = [hf_hub_download(cfg.dataset_repo, n, repo_type="dataset",
                                 revision=cfg.dataset_revision) for n in names]
        ds = load_dataset("parquet", data_files=paths, split="train")
        episodes, seen = [], set()
        for r in ds:
            cid = r["conv_id"]
            if cid in seen:
                continue
            seen.add(cid)
            text, label = r["prompt"].strip(), r["context"].strip()
            if text and label:
                episodes.append({"text": text, "label": label})
        return episodes

    ed_train_full = load_ed_episodes("train")
    ed_val_full = load_ed_episodes("validation")
    ed_test_full = load_ed_episodes("test")
    print(f"ED episodes: train {len(ed_train_full)} | "
          f"val {len(ed_val_full)} | test {len(ed_test_full)}")

    emotion_labels = sorted({e["label"] for e in ed_train_full})
    assert len(emotion_labels) == 32, f"ожидалось 32 эмоции, есть {len(emotion_labels)}"

    def _norm(s):
        return re.sub(r"[^a-z ]", "", s.lower()).strip()

    EMOTION_BY_NORM = {_norm(l): l for l in emotion_labels}

    def match_emotion(text):
        g = _norm(text)
        if not g:
            return None
        first = g.split()[0]
        if first in EMOTION_BY_NORM:
            return EMOTION_BY_NORM[first]
        if g in EMOTION_BY_NORM:
            return EMOTION_BY_NORM[g]
        for nl, l in EMOTION_BY_NORM.items():
            if g.startswith(nl):
                return l
        for nl, l in EMOTION_BY_NORM.items():
            if nl in g:
                return l
        return None

    LABEL_TOK_IDS = {
        lab: tok(" " + lab, add_special_tokens=False,
                 truncation=True, max_length=cfg.max_target_len)["input_ids"]
        for lab in emotion_labels
    }

    # D1.6: первые токены меток — для вербализатор-скоринга
    # (протокол статьи: top-scoring output token). Проверяем
    # единственность: две метки с одним первым токеном неразличимы.
    LABEL_FIRST_TOKEN = {lab: ids[0] for lab, ids in LABEL_TOK_IDS.items()}
    _firsts = list(LABEL_FIRST_TOKEN.values())
    assert len(set(_firsts)) == len(_firsts), (
        "коллизия первых токенов меток — вербализатор неразличим: " +
        str([l for l in LABEL_FIRST_TOKEN
             if list(LABEL_FIRST_TOKEN.values()).count(LABEL_FIRST_TOKEN[l]) > 1])
    )
    _multi = [l for l in emotion_labels if len(LABEL_TOK_IDS[l]) > 1]
    print(f"Вербализатор: все 32 первых токена уникальны; "
          f"мультитокенных меток: {len(_multi)} {_multi}")

    # D1.6: ЖЁСТКИЕ негативы — группы путаемых эмоций (критика D1.5:
    # случайная чужая метка из 31 слишком легка; негатив должен быть
    # реальной конкурирующей альтернативой)
    CONFUSION_GROUPS = [
        {"afraid", "terrified", "anxious", "apprehensive"},
        {"sad", "disappointed", "devastated", "lonely"},
        {"angry", "furious", "annoyed", "disgusted"},
        {"surprised", "anticipating", "excited", "joyful"},
        {"guilty", "ashamed", "embarrassed"},
        {"hopeful", "faithful", "trusting", "prepared", "confident", "proud"},
        {"caring", "grateful", "impressed"},
        {"sentimental", "nostalgic", "content"},
        {"jealous"},
    ]
    LABEL_GROUP = {}
    for g in CONFUSION_GROUPS:
        for lab in g:
            if lab in emotion_labels:
                LABEL_GROUP[lab] = g
    print(f"Группы путаемости покрывают {len(LABEL_GROUP)}/32 меток")

    _neg_rng = random.Random(1234)

    def sample_wrong_label(true_label):
        """Жёсткий негатив: предпочтительно из группы путаемости."""
        group = LABEL_GROUP.get(true_label)
        if group and len(group) > 1:
            pool = [l for l in group if l != true_label]
            return _neg_rng.choice(pool)
        pool = [l for l in emotion_labels if l != true_label]
        return _neg_rng.choice(pool)

    raw_rl    = HFDataset.from_list(ed_train_full[:cfg.rl_subset_size])
    raw_train = HFDataset.from_list(ed_train_full[cfg.rl_subset_size:
                                                  cfg.rl_subset_size + cfg.train_size])
    raw_val   = HFDataset.from_list(ed_val_full[:cfg.val_size])
    raw_test  = HFDataset.from_list(ed_test_full[:cfg.test_size])

    assert cfg.rl_subset_size + cfg.train_size <= len(ed_train_full), \
        "hold-out + train не помещаются в датасет"

    print(f"Эмоции (32): {emotion_labels}")
else:
    raw = load_dataset(cfg.dataset_name)
    print("Splits:", list(raw.keys()))
    raw_rl    = raw["train"].select(range(0, cfg.rl_subset_size))
    raw_train = raw["train"].select(range(cfg.rl_subset_size,
                                          cfg.rl_subset_size + cfg.train_size))
    raw_val   = raw["validation"].select(range(cfg.val_size))
    raw_test  = raw["test"].select(range(cfg.test_size))
    assert cfg.rl_subset_size + cfg.train_size <= len(raw["train"])

def tokenize_gen(examples):
    all_input_ids = []
    all_attention_mask = []
    all_labels = []
    all_neg_input_ids = []
    all_neg_attention_mask = []
    all_neg_labels = []

    max_total = cfg.max_total_len
    tgt_prefix = " " if cfg.task_kind == "emotion_cls" else ""
    add_eos = cfg.task_kind != "emotion_cls"  # D1.1b: без EOS в cls

    for s, t in zip(examples[cfg.text_column], examples[cfg.target_column]):
        src = cfg.prompt_template.format(source=s)

        src_ids = tok(
            src,
            add_special_tokens=False,
            truncation=True,
            max_length=cfg.max_source_len,
        )["input_ids"]

        tgt_max = max(1, cfg.max_target_len - 1) if add_eos else cfg.max_target_len
        tgt_ids = tok(
            tgt_prefix + t,
            add_special_tokens=False,
            truncation=True,
            max_length=tgt_max,
        )["input_ids"]
        if add_eos:
            tgt_ids = tgt_ids + [tok.eos_token_id]

        if len(src_ids) == 0:
            src_ids = [tok.pad_token_id]

        if len(src_ids) + len(tgt_ids) > max_total:
            src_ids = src_ids[:max(1, max_total - len(tgt_ids))]

        input_ids = src_ids + tgt_ids
        labels = [-100] * len(src_ids) + tgt_ids

        all_input_ids.append(input_ids)
        all_attention_mask.append([1] * len(input_ids))
        all_labels.append(labels)

        # D1.6: ЖЁСТКИЙ негатив wrong-label — (ситуация, путаемая эмоция)
        if cfg.task_kind == "emotion_cls":
            wrong_ids = list(LABEL_TOK_IDS[sample_wrong_label(t)])
            neg_ids = src_ids + wrong_ids
            all_neg_input_ids.append(neg_ids)
            all_neg_attention_mask.append([1] * len(neg_ids))
            all_neg_labels.append([-100] * len(src_ids) + wrong_ids)

    out = {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    }
    if cfg.task_kind == "emotion_cls":
        out.update({
            "neg_input_ids": all_neg_input_ids,
            "neg_attention_mask": all_neg_attention_mask,
            "neg_labels": all_neg_labels,
        })
    return out

train_ds = raw_train.map(tokenize_gen, batched=True, remove_columns=raw_train.column_names)
val_ds = raw_val.map(tokenize_gen, batched=True, remove_columns=raw_val.column_names)
test_ds = raw_test.map(tokenize_gen, batched=True, remove_columns=raw_test.column_names)
rl_ds = raw_rl.map(tokenize_gen, batched=True, remove_columns=raw_rl.column_names)

rl_ds = rl_ds.add_column("ref_idx", list(range(len(rl_ds))))

rl_references = raw_rl[cfg.target_column]
val_references = raw_val[cfg.target_column]
test_references = raw_test[cfg.target_column]

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")
rl_ds.set_format("torch")

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} | "
      f"RL hold-out: {len(rl_ds)} (не входит в train)")
assert (train_ds[0]["labels"] != -100).sum() > 0, "нет супервизируемых токенов"
if cfg.task_kind == "emotion_cls":
    print("Sample NEG (последние 10 токенов):", train_ds[0]["neg_labels"][-10:])


ED episodes: train 17844 | val 2763 | test 2542
Вербализатор: все 32 первых токена уникальны; мультитокенных меток: 1 ['apprehensive']
Группы путаемости покрывают 32/32 меток
Эмоции (32): ['afraid', 'angry', 'annoyed', 'anticipating', 'anxious', 'apprehensive', 'ashamed', 'caring', 'confident', 'content', 'devastated', 'disappointed', 'disgusted', 'embarrassed', 'excited', 'faithful', 'furious', 'grateful', 'guilty', 'hopeful', 'impressed', 'jealous', 'joyful', 'lonely', 'nostalgic', 'prepared', 'proud', 'sad', 'sentimental', 'surprised', 'terrified', 'trusting']


Map: 100%|██████████| 600/600 [00:00<00:00, 8655.63 examples/s]

Train: 6000 | Val: 200 | Test: 500 | RL hold-out: 600 (не входит в train)
Sample NEG (последние 10 токенов): tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 68244])


In [5]:
# ============================================================
# CELL 5: Reward function
# D1 (emotion_cls): accuracy матчинга сгенерированного слова к 32 меткам
# summarization (наследие): ROUGE-L против референса
# ============================================================
rouge_metric = evaluate.load("rouge")

@torch.no_grad()
def compute_generation_reward(model, input_ids, attention_mask, references, tokenizer,
                              task_cfg=None):
    """Reward на ДАННОМ примере против его референса. Greedy ->
    детерминированный reward (аналог accuracy в статье).

    Для task_kind="emotion_cls": references = строки-эмоции,
    reward = доля примеров, где match_emotion(генерация) == метка.
    Для "summarization": reward = ROUGE-L.
    """
    # D2.2: task_cfg — конфиг ЗАДАЧИ (глобальный cfg может быть чужим,
    # например D1-классификацией); по умолчанию прежнее поведение
    c = task_cfg if task_cfg is not None else cfg
    model.eval()

    old_use_cache = getattr(model.config, "use_cache", True)
    model.config.use_cache = True

    try:
        gen_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=c.gen_max_new_tokens,
            do_sample=c.gen_do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if c.gen_do_sample:
            gen_kwargs.update(temperature=c.gen_temperature, top_p=c.gen_top_p)

        gen_ids = model.generate(**gen_kwargs)
    finally:
        model.config.use_cache = old_use_cache

    new_tokens = gen_ids[:, input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    generated_texts = [x.strip() if x.strip() else " " for x in generated_texts]

    assert len(generated_texts) == len(references), \
        f"predictions/references mismatch: {len(generated_texts)} vs {len(references)}"

    if c.task_kind == "emotion_cls":
        reward = float(np.mean([
            match_emotion(g) == r
            for g, r in zip(generated_texts, references)
        ]))
    else:
        results = rouge_metric.compute(
            predictions=generated_texts,
            references=references,
            rouge_types=["rougeL"],
        )
        reward = float(results["rougeL"])

    model.train()
    return reward, generated_texts

print("Reward function defined: accuracy (emotion_cls) / ROUGE-L (summarization).")


Reward function defined: accuracy (emotion_cls) / ROUGE-L (summarization).


In [6]:
# ============================================================
# CELL 6: TwoPhaseTrainerGen — адаптация Algorithm 1
# D1.5: contrast_mode = "mask" | "wrong_label"; contrast_from_start
# D1.6: RL reward на ЧИСТОМ входе (strip золотой метки — баг эха)
# ============================================================
class TwoPhaseTrainerGen(Trainer):
    def __init__(
        self,
        model=None,
        args=None,
        phase1_epochs: int = 3,
        alpha: float = 1.0,
        beta: float = 0.1,
        gamma: float = 2e-5,
        gamma_auto: bool = True,
        gamma_target_frac: float = 0.3,
        sigma: float = 0.02,
        contrast_mode: str = "mask",
        contrast_from_start: bool = False,
        contrastive_dropout: float = 0.1,
        contrastive_tau: float = 0.5,
        k_negatives: int = 1,
        rl_interval: int = 50,
        rl_num_batches: int = 16,
        rl_subset_size: int = 200,
        use_reward_baseline: bool = True,
        phase2_lr: float = 2e-4,
        divergence_ce_threshold: float = 6.0,
        guard_start_step: int = 200,
        rl_dataset=None,
        rl_references=None,
        tokenizer=None,
        task_cfg=None,
        contrast_neg_in_graph: bool = False,
        reward_baseline_beta: float = 0.9,
        **kwargs
    ):
        self.phase1_epochs = phase1_epochs
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.gamma_auto = gamma_auto
        self.gamma_target_frac = gamma_target_frac
        self.sigma = sigma
        self.contrast_mode = contrast_mode
        self.contrast_from_start = contrast_from_start
        self.contrastive_dropout = contrastive_dropout
        self.contrastive_tau = contrastive_tau
        self.k_negatives = k_negatives
        self.rl_interval = rl_interval
        self.rl_num_batches = rl_num_batches
        self.rl_subset_size = rl_subset_size
        self.use_reward_baseline = use_reward_baseline
        self.phase2_lr = phase2_lr
        self.divergence_ce_threshold = divergence_ce_threshold
        self.guard_start_step = guard_start_step
        self.rl_dataset = rl_dataset
        self.rl_references = rl_references
        self.tokenizer = tokenizer
        self.task_cfg = task_cfg          # D2.2: конфиг задачи для reward
        self.rl_reward_history = []       # D2.2: трекинг наград по шагам
        self.contrast_neg_in_graph = contrast_neg_in_graph  # D2.3
        self.reward_baseline_beta = reward_baseline_beta    # D2.2b

        # Флаги состояний
        self._phase2_lr_reset_done = False
        self._gamma_calibrated = False
        self._last_rl_step = -1
        self._last_log_step = -1
        self._reward_baseline = 0.0
        self._ce_ema = None
        self._divergence_reported = False

        super().__init__(model=model, args=args, **kwargs)

        # Параметры префикса собираем ПОСЛЕ super().__init__
        self.prefix_params = [
            (n, p) for n, p in self.model.named_parameters()
            if "prompt_encoder" in n and p.requires_grad
        ]
        print(f"[Trainer] Prefix params collected: {len(self.prefix_params)}")

        # Цели контраста (для mode="mask") — ТОЛЬКО embedding-слой
        self.contrast_params = [
            (n, p) for n, p in self.prefix_params if "embedding" in n
        ] or self.prefix_params
        print(f"[Trainer] Contrast: mode={self.contrast_mode}, "
              f"from_start={self.contrast_from_start}, beta={self.beta}")

        if self.rl_dataset is not None:
            self._rl_loader = DataLoader(
                self.rl_dataset,
                batch_size=self.args.per_device_train_batch_size,
                collate_fn=self.data_collator,
            )
            self._rl_iter = iter(self._rl_loader)

    # ----------------------------------------------------------
    # Утилиты
    # ----------------------------------------------------------
    @staticmethod
    def _model_inputs(inputs):
        """Только те ключи, которые понимает model.forward. Коллатор
        дополнительно кладёт neg_*/ref_idx — их в модель нельзя."""
        return {k: v for k, v in inputs.items()
                if k in ("input_ids", "attention_mask", "labels")}

    def _next_rl_batch(self):
        try:
            return next(self._rl_iter)
        except StopIteration:
            self._rl_iter = iter(self._rl_loader)
            return next(self._rl_iter)

    def _restore_params(self, params, snapshot):
        with torch.no_grad():
            for n, p in params:
                p.data.copy_(snapshot[n])

    def _custom_scale(self):
        """ЕДИНЫЙ множитель для ВСЕГО лосса, включая CE (урок ит. 2
        исследования-1: иначе эффективный LR x gradient_accumulation)."""
        trainer_divides = not getattr(self, "model_accepts_loss_kwargs", False)
        return 1.0 if trainer_divides else 1.0 / self.args.gradient_accumulation_steps

    def _reset_lr_for_phase2(self):
        if self.optimizer is None or self.lr_scheduler is None:
            return
        remaining = max(1, self.state.max_steps - self.state.global_step)
        for pg in self.optimizer.param_groups:
            pg["lr"] = self.phase2_lr
            pg["initial_lr"] = self.phase2_lr
        self.lr_scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, num_warmup_steps=0, num_training_steps=remaining,
        )
        print(f"\n[Phase 2] LR reset to {self.phase2_lr}, "
              f"new cosine over remaining {remaining} steps\n")

    def _pool_hidden(self, hidden, labels):
        """Пуллинг по токенам таргета (labels != -100): для позитива —
        токены верной метки, для негатива — токены чужой метки."""
        if hidden.size(1) != labels.size(1):
            hidden = hidden[:, -labels.size(1):, :]
        tgt_mask = (labels != -100).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * tgt_mask).sum(1) / tgt_mask.sum(1).clamp(min=1)
        return F.normalize(pooled.float(), dim=-1)

    # ----------------------------------------------------------
    # Contrastive loss — два режима негативов
    # ----------------------------------------------------------
    def _contrastive_loss(self, model, inputs, h_pos):
        """L = log1 + sum_j exp((sim(h_i, h_j^-) - 1) / tau))

        mode="mask" (рецепт статьи): h_j^- — представление ТОГО ЖЕ входа
        под замаскированным embedding-слоем промпта. Позитив тривиален
        (sim(h,h)=1 — вырожденность Eq. 3, см. FINAL_ANALYSIS §4).

        mode="wrong_label" (ядро темы): негатив — (ситуация, ЖЁСТКАЯ
        чужая эмоция из группы путаемости) — реальная конкурирующая
        альтернатива."""
        if self.k_negatives <= 0:
            return h_pos.new_zeros(())

        h_negs = []

        # D2: негативы из neg-колонок, собранных при токенизации
        # (neg = wrong-label для cls; negoff*/neginc для response_gen).
        # D2.3: offtopic4 = k=4 негативов (Σ_j как в Eq.3 статьи);
        # oftopic_grad = градиент и через негатив (симметричный апдейт)
        COLUMN_MODES = {
            "wrong_label": ["neg"],
            "offtopic": ["negoff"],
            "offtopic_grad": ["negoff"],
            "offtopic4": ["negoff", "negoff2", "negoff3", "negoff4"],
            "incoherent": ["neginc"],
        }
        if self.contrast_mode in COLUMN_MODES:
            for pfx in COLUMN_MODES[self.contrast_mode]:
                neg_fwd = {
                    "input_ids": inputs[f"{pfx}_input_ids"],
                    "attention_mask": inputs[f"{pfx}_attention_mask"],
                }
                if getattr(self, "contrast_neg_in_graph", False):
                    out_neg = model(**neg_fwd, output_hidden_states=True)
                else:
                    with torch.no_grad():
                        out_neg = model(**neg_fwd, output_hidden_states=True)
                h_negs.append(self._pool_hidden(
                    out_neg.hidden_states[-1], inputs[f"{pfx}_labels"]))
                del out_neg
        else:
            neg_inputs = {k: v for k, v in self._model_inputs(inputs).items()
                          if k != "labels"}
            labels = inputs["labels"]
            original_data = {n: p.data.clone() for n, p in self.contrast_params}
            try:
                for _ in range(self.k_negatives):
                    for n, p in self.contrast_params:
                        mask = (torch.rand_like(p.float()) > self.contrastive_dropout).to(p.dtype)
                        p.data.copy_((original_data[n] * mask).to(p.dtype))
                    with torch.no_grad():
                        out_neg = model(**neg_inputs, output_hidden_states=True)
                    h_neg = self._pool_hidden(out_neg.hidden_states[-1], labels)
                    h_negs.append(h_neg)
                    del out_neg
                    self._restore_params(self.contrast_params, original_data)
            finally:
                self._restore_params(self.contrast_params, original_data)

        h_negs = torch.stack(h_negs, dim=1)  # (B, K, D)

        tau = self.contrastive_tau
        neg_scores = torch.einsum("bd,bkd->bk", h_pos, h_negs) / tau
        loss = torch.log1p(torch.exp(neg_scores - 1.0 / tau).sum(dim=1)).mean()
        return loss

    # ----------------------------------------------------------
    # RL loss (REINFORCE, Eq. 4 статьи)
    # ----------------------------------------------------------
    def _rl_loss(self, model):
        log_probs = []
        noisy_values = {}
        for n, p in self.prefix_params:
            p32 = p.float()
            eps = torch.randn_like(p32) * self.sigma
            noisy = (p32.detach() + eps).detach()
            log_probs.append(
                Normal(loc=p32, scale=self.sigma).log_prob(noisy).sum()
            )
            noisy_values[n] = noisy

        original_data = {n: p.data.clone() for n, p in self.prefix_params}
        rewards = []
        last_texts, last_refs = None, None
        try:
            for n, p in self.prefix_params:
                p.data.copy_(noisy_values[n].to(p.dtype))

            for _ in range(self.rl_num_batches):
                rl_batch = self._next_rl_batch()
                idx = rl_batch.pop("ref_idx", None)
                if idx is None:
                    raise RuntimeError(
                        "ref_idx потерян коллатором: causal_lm_collator обязан "
                        "пробрасывать ref_idx"
                    )
                # D1.6: ЧИСТЫЙ reward — отрезаем золотую метку до generate
                # (баг эха: иначе reward мерил повторение подсказанной метки)
                src_ids, src_mask = strip_label_tokens(rl_batch)
                batch = {
                    "input_ids": src_ids.to(model.device),
                    "attention_mask": src_mask.to(model.device),
                }
                refs = [self.rl_references[i] for i in idx.tolist()]
                r, texts = compute_generation_reward(
                    model, batch["input_ids"], batch["attention_mask"],
                    refs, self.tokenizer, task_cfg=self.task_cfg,
                )
                rewards.append(r)
                last_texts, last_refs = texts, refs
            reward = float(np.mean(rewards))
            self.rl_reward_history.append(
                {"step": int(self.state.global_step), "reward": reward})
        finally:
            self._restore_params(self.prefix_params, original_data)

        if self.use_reward_baseline:
            advantage = reward - self._reward_baseline
            _beta_b = self.reward_baseline_beta   # D2.2b: инерция EMA
            self._reward_baseline = (
                _beta_b * self._reward_baseline + (1 - _beta_b) * reward)
        else:
            advantage = reward

        raw_rl = -advantage * torch.stack(log_probs).sum()

        if self.gamma_auto and not self._gamma_calibrated:
            raw_abs = abs(float(raw_rl.detach()))
            if raw_abs < 1e-6 or self._ce_ema is None:
                print("[RL] gamma calibration postponed")
            else:
                accum = self.args.gradient_accumulation_steps
                self.gamma = float(np.clip(
                    self.gamma_target_frac * self._ce_ema * accum / raw_abs,
                    1e-8, 1.0,
                ))
                self._gamma_calibrated = True
                print(f"[RL] gamma auto-calibrated to {self.gamma:.3e}")

        loss_rl = self.gamma * raw_rl

        if last_texts is not None:
            print(f"[RL sample] REF: {last_refs[0][:110]!r}")
            print(f"[RL sample] GEN: {last_texts[0][:110]!r}")
        print(f"[RL step {self.state.global_step}] reward={reward:.4f} "
              f"adv={advantage:+.4f} gamma={self.gamma:.2e}")
        return loss_rl, reward

    # ----------------------------------------------------------
    # Основной лосс
    # ----------------------------------------------------------
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        current_epoch = int(self.state.epoch) if self.state.epoch else 0
        in_phase2 = current_epoch >= self.phase1_epochs

        if in_phase2 and not self._phase2_lr_reset_done:
            self._phase2_lr_reset_done = True
            self._reset_lr_for_phase2()

        use_contrast = self.beta != 0.0 and (in_phase2 or self.contrast_from_start)

        model_inputs = self._model_inputs(inputs)
        outputs = model(**model_inputs, output_hidden_states=use_contrast)
        loss_mle = outputs.loss

        ce_now = loss_mle.item()
        self._ce_ema = ce_now if self._ce_ema is None else 0.95 * self._ce_ema + 0.05 * ce_now

        if (self.state.global_step >= self.guard_start_step
                and self._ce_ema > self.divergence_ce_threshold
                and not self._divergence_reported):
            self._divergence_reported = True
            print(f"\n!!! CE/token EMA = {self._ce_ema:.2f} > "
                  f"{self.divergence_ce_threshold} — расходимость. "
                  f"Останавливаю обучение (шаг {self.state.global_step}).\n")
            self.control.should_training_stop = True

        scale = self._custom_scale()

        loss_contrast = loss_mle.new_zeros(())
        if use_contrast:
            h_pos = self._pool_hidden(outputs.hidden_states[-1],
                                      model_inputs["labels"])
            loss_contrast = self._contrastive_loss(model, inputs, h_pos)

        loss_rl = loss_mle.new_zeros(())
        reward_val = 0.0
        rl_enabled = (self.gamma != 0.0 or self.gamma_auto) and self.rl_dataset is not None
        if (
            rl_enabled
            and self.state.global_step % self.rl_interval == 0
            and self._last_rl_step != self.state.global_step
        ):
            self._last_rl_step = self.state.global_step
            loss_rl, reward_val = self._rl_loss(model)

        if (model.training and self.state.global_step % 10 == 0
                and self._last_log_step != self.state.global_step):
            self._last_log_step = self.state.global_step
            print(f"[Step {self.state.global_step}] CE/token (EMA): {self._ce_ema:.3f} | "
                  f"Contrast[{self.contrast_mode}]: {loss_contrast.item():.3f} | "
                  f"RL: {loss_rl.item():.3f} | Reward: {reward_val:.4f}")

        loss = scale * (self.alpha * loss_mle + self.beta * loss_contrast + loss_rl)
        return (loss, outputs) if return_outputs else loss

print("TwoPhaseTrainerGen defined ✓")


TwoPhaseTrainerGen defined ✓


In [7]:
# ============================================================
# CELL 7: collator + ЧИСТЫЙ eval (фикс бага эха) + коллбэки
#
# БАГ ПРОТОКОЛА D1.x (найден при разборе критики): вход для generate
# СОДЕРЖАЛ золотую метку в конце -> модель продолжала её, и
# «предсказанием» было ЭХО метки. Канарейка D1.5 это показывает
# буквально: «surprised surprised surprised…» — повтор золотой метки.
# Отсюда подозрительно высокие accuracy обученных моделей (0.45–0.65)
# при чистом zero-shot 0.18. Фикс: перед generate отрезаем токены
# метки (labels != -100 считаются от КОНЦА последовательности).
# ============================================================
import json as _json
import json
import torch.nn.functional as F  # ре-импорт: защита от затенения имени F
                                  # (баг: ячейка BERTScore писала P, R, F = ...)


def strip_label_tokens(batch):
    """Отрезает токены метки в конце каждого примера и заново
    лево-пэддит батч. Возвращает (input_ids, attention_mask)
    src-only — модель должна ПРЕДСКАЗАТЬ метку, а не прочитать."""
    labels = batch["labels"]
    n_lab = (labels != -100).sum(dim=1)
    L = batch["input_ids"].size(1)
    srcs = [batch["input_ids"][i, :L - int(n_lab[i])]
            for i in range(batch["input_ids"].size(0))]
    # Маску берём ТЕМ ЖЕ срезом из оригинала: в отрезанном куске
    # остаются исходные лево-паддинговые позиции с маской 0
    # (юнит-тест поймал баг: маска «все единицы» ломала позициями)
    ams = [batch["attention_mask"][i, :L - int(n_lab[i])]
           for i in range(batch["input_ids"].size(0))]
    max_len = max(s.size(0) for s in srcs)
    input_ids = torch.stack([
        s if s.size(0) == max_len
        else F.pad(s, (max_len - s.size(0), 0), value=tok.pad_token_id)
        for s in srcs
    ])
    attention_mask = torch.stack([
        m if m.size(0) == max_len
        else F.pad(m, (max_len - m.size(0), 0), value=0)
        for m in ams
    ])
    return input_ids, attention_mask


@torch.no_grad()
def verbalizer_eval(model, dataset, references, tokenizer, batch_size=16):
    """Вербализатор-скоринг (протокол статьи: top-scoring output token):
    ОДИН forward на src-only входе, argmax по первым токенам 32 меток
    в последней позиции. Детерминированно, без генерации.
    Возвращает (accuracy, correct_vec)."""
    if cfg.task_kind != "emotion_cls":
        return None, None
    model.eval()
    order_labels = sorted(LABEL_FIRST_TOKEN.keys())
    tok_ids = torch.tensor([LABEL_FIRST_TOKEN[l] for l in order_labels],
                           device=model.device)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=causal_lm_collator)
    correct = []
    idx = 0
    for batch in loader:
        input_ids, attention_mask = strip_label_tokens(batch)
        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)
        logits = model(input_ids=input_ids,
                       attention_mask=attention_mask).logits[:, -1, :]
        pred_idx = logits[:, tok_ids].argmax(dim=-1).tolist()
        for pi in pred_idx:
            correct.append(int(order_labels[pi] == references[idx]))
            idx += 1
    return float(np.mean(correct)), correct


def causal_lm_collator(features):
    pad_token_id = tok.pad_token_id

    def to_1d_tensor(x, dtype=torch.long):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=dtype)
        if x.dim() > 1:
            x = x.squeeze(0)
        return x

    def pad_batch(seqs, value=0):
        max_len = max(s.size(0) for s in seqs)
        return torch.stack([
            s if s.size(0) == max_len
            else F.pad(s, (max_len - s.size(0), 0), value=value)
            for s in seqs
        ])

    # ПАДДИНГ = 0 ДЛЯ ОБОИХ (легаси всех завершённых прогонов).
    # Эмпирически доказано (журнал §6.3): в этом стеке значение пэда
    # input_ids ВЛИЯЕТ на логиты реальных позиций (max diff 6.2 даже
    # при нулевой маске) — «гигиена» pad_token_id НЕ числово-нейтральна,
    # а attention_mask с ненулевыми падами ломает маскирование вовсе.
    # Поэтому: (1) оставляем 0/0 как во всех проведённых экспериментах;
    # (2) замена на pad_token_id возможна ТОЛЬКО при полном перезапуске.
    # Известная историческая особенность: eval-пути (strip_label_tokens,
    # zero-shot через токенизатор) паддят pad_token_id — train/eval
    # рассогласование существовало ВСЕГДА, одинаково для всех конфигов
    # (парные выводы не затронуты), задокументировано в §6.3.
    out = {
        "input_ids": pad_batch([to_1d_tensor(f["input_ids"]) for f in features]),
        "attention_mask": pad_batch(
            [to_1d_tensor(f["attention_mask"]) for f in features]),
    }
    # labels: левый паддинг значением -100
    lab_list = []
    max_len = out["input_ids"].size(1)
    for f in features:
        labs = to_1d_tensor(f["labels"])
        pad_len = max_len - labs.size(0)
        lab_list.append(labs if pad_len == 0 else F.pad(labs, (pad_len, 0), value=-100))
    out["labels"] = torch.stack(lab_list)

    # Негативные последовательности: neg (wrong-label, cls),
    # negoff / negoff2..4 / neginc (D2; D2.3 — динамическое
    # обнаружение любых *_input_ids-колонок негативов)
    neg_pfxs = sorted({k[:-len("_input_ids")] for k in features[0]
                       if k.endswith("_input_ids") and k != "input_ids"})
    for pfx in neg_pfxs:
        if f"{pfx}_attention_mask" in features[0]:
            n_ids = [to_1d_tensor(f[f"{pfx}_input_ids"]) for f in features]
            n_mask = [to_1d_tensor(f[f"{pfx}_attention_mask"]) for f in features]
            n_labs = [to_1d_tensor(f[f"{pfx}_labels"]) for f in features]
            max_len = max(s.size(0) for s in n_ids)
            out[f"{pfx}_input_ids"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=pad_token_id)
                for s in n_ids])
            out[f"{pfx}_attention_mask"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=0)
                for s in n_mask])
            out[f"{pfx}_labels"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=-100)
                for s in n_labs])

    if "ref_idx" in features[0]:
        out["ref_idx"] = torch.tensor(
            [int(f["ref_idx"]) for f in features], dtype=torch.long
        )

    return out


class Phase1CanaryCallback(TrainerCallback):
    """Каждую эпоху печатает ЧИСТЫЕ генерации (src-only) на val;
    если ВСЕ пустые — останавливает обучение."""

    def __init__(self, canary_ds, tokenizer, collator, first_epoch=2,
                 num_examples=4, max_new_tokens=6):
        n = min(num_examples, len(canary_ds))
        self.loader = DataLoader(
            canary_ds.select(range(n)), batch_size=1, collate_fn=collator
        )
        self.tokenizer = tokenizer
        self.first_epoch = first_epoch
        self.max_new_tokens = max_new_tokens
        self.stopped = False

    @torch.no_grad()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None or state.epoch < self.first_epoch or self.stopped:
            return

        was_training = model.training
        model.eval()
        old_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True

        texts = []
        try:
            for batch in self.loader:
                input_ids, attention_mask = strip_label_tokens(batch)
                input_ids = input_ids.to(model.device)
                attention_mask = attention_mask.to(model.device)
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                gen = self.tokenizer.decode(
                    out[0, input_ids.shape[1]:], skip_special_tokens=True
                ).strip()
                texts.append(gen if gen else "<EMPTY>")
        finally:
            model.config.use_cache = old_use_cache
            if was_training:
                model.train()

        print(f"[Canary epoch {state.epoch:.0f}] " + " | ".join(t[:80] for t in texts))

        if all(t == "<EMPTY>" for t in texts):
            self.stopped = True
            print("!!! Canary: ВСЕ генерации пустые — модель не обучилась.\n"
                  "    Останавливаю обучение.")
            control.should_training_stop = True


class BestEpochValMetricCallback(TrainerCallback):
    """Раз в эпоху: ЧИСТАЯ метрика (src-only генерация + матчинг)
    на N val-примерах + сохранение ЛУЧШЕГО адаптера в best_prefix.pt."""

    def __init__(self, val_ds, val_references, tokenizer, collator,
                 output_dir, task_kind="emotion_cls",
                 num_examples=200, max_new_tokens=6):
        n = min(num_examples, len(val_ds), len(val_references))
        self.loader = DataLoader(
            val_ds.select(range(n)), batch_size=8, collate_fn=collator
        )
        self.references = list(val_references[:n])
        self.tokenizer = tokenizer
        self.output_dir = output_dir
        self.task_kind = task_kind
        self.metric_name = "accuracy" if task_kind == "emotion_cls" else "rougeL"
        self.max_new_tokens = max_new_tokens
        self.rouge = evaluate.load("rouge") if task_kind != "emotion_cls" else None
        self.best = -1.0
        self.best_epoch = None
        self.history = []
        self._collapse_reported = False

    @torch.no_grad()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return

        was_training = model.training
        model.eval()
        old_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True

        preds = []
        try:
            for batch in self.loader:
                input_ids, attention_mask = strip_label_tokens(batch)
                input_ids = input_ids.to(model.device)
                attention_mask = attention_mask.to(model.device)
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                for j in range(input_ids.size(0)):
                    pred = self.tokenizer.decode(
                        out[j, input_ids.shape[1]:], skip_special_tokens=True
                    ).strip()
                    preds.append(pred if pred else " ")
        finally:
            model.config.use_cache = old_use_cache
            if was_training:
                model.train()

        if self.task_kind == "emotion_cls":
            score = float(np.mean([
                match_emotion(p) == r for p, r in zip(preds, self.references)
            ]))
        else:
            score = float(self.rouge.compute(
                predictions=preds, references=self.references,
                rouge_types=["rougeL"],
            )["rougeL"])
        self.history.append({"epoch": round(state.epoch, 2), "value": score})

        if score > self.best:
            self.best = score
            self.best_epoch = round(state.epoch, 2)
            os.makedirs(self.output_dir, exist_ok=True)
            torch.save(
                {n: p.data.detach().cpu().clone()
                 for n, p in model.named_parameters() if p.requires_grad},
                os.path.join(self.output_dir, "best_prefix.pt"),
            )

        # D2: стоп коллапса генерации. Урок ARTICLE s44 (§4.16): CE-плато
        # скрывает деградацию генерации — guard по лоссу её не видит,
        # канарейка видит только ПУСТЫЕ генерации. Здесь: глубокое
        # падение val-метрики относительно лучшей = остановка.
        if (len(self.history) >= 3 and score < 0.5 * self.best
                and not self._collapse_reported):
            self._collapse_reported = True
            print(f"\n!!! Коллапс: val {self.metric_name}={score:.4f} < "
                  f"0.5 x best ({self.best:.4f}) на эпохе {state.epoch:.0f} "
                  f"— останавливаю обучение.\n")
            control.should_training_stop = True
        elif (len(self.history) >= 3 and score < 0.75 * self.best
                and not getattr(self, "_softdeg_warned", False)):
            # критика-2: мягкая деградация (0.5x < val < 0.75x best) —
            # только предупреждение: агрессивный стоп опасен (MLE s44
            # D2.1 восстановился с 0.026 на e1 до 0.176 на e2)
            self._softdeg_warned = True
            print(f"[SoftDegradation] val {self.metric_name}={score:.4f} < "
                  f"0.75 x best ({self.best:.4f}) — предупреждение, "
                  f"БЕЗ остановки")

        with open(os.path.join(self.output_dir, "val_metric_history.json"), "w") as f:
            _json.dump(
                {"metric": self.metric_name, "history": self.history,
                 "best": self.best, "best_epoch": self.best_epoch},
                f, indent=2,
            )

        print(f"[ValMetric epoch {state.epoch:.0f}] {self.metric_name}={score:.4f} | "
              f"best={self.best:.4f} @ epoch {self.best_epoch:.0f}")

    def on_train_end(self, args, state, control, **kwargs):
        print(f"[ValMetric] Итог: best {self.metric_name}={self.best:.4f} @ epoch "
              f"{self.best_epoch:.0f} -> best_prefix.pt")


def prepare_trainer(model, train_ds, val_ds, rl_ds, rl_references, tokenizer, cfg,
                    val_references=None):
    """Конфигурация тренера полностью из cfg — включая output_dir."""

    def compute_metrics(eval_pred):
        return {}

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.total_epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler,
        warmup_steps=cfg.warmup_steps,
        logging_steps=cfg.logging_steps,
        eval_strategy=cfg.eval_strategy,
        save_strategy=cfg.save_strategy,
        save_total_limit=cfg.save_total_limit,
        load_best_model_at_end=cfg.load_best_model_at_end,
        bf16=cfg.bf16,
        fp16=cfg.fp16,
        gradient_checkpointing=cfg.gradient_checkpointing,
        report_to=cfg.report_to,
        remove_unused_columns=cfg.remove_unused_columns,
        dataloader_pin_memory=cfg.dataloader_pin_memory,
        seed=cfg.seed,
        prediction_loss_only=True,
        max_grad_norm=cfg.max_grad_norm,
    )

    canary = Phase1CanaryCallback(
        canary_ds=val_ds,
        tokenizer=tokenizer,
        collator=causal_lm_collator,
        first_epoch=2,
        max_new_tokens=cfg.eval_max_new_tokens,
    )

    callbacks = [canary]

    if val_references is not None:
        callbacks.append(BestEpochValMetricCallback(
            val_ds=val_ds,
            val_references=val_references,
            tokenizer=tokenizer,
            collator=causal_lm_collator,
            output_dir=cfg.output_dir,
            task_kind=cfg.task_kind,
            num_examples=cfg.val_rouge_examples,
            max_new_tokens=cfg.eval_max_new_tokens,
        ))

    trainer = TwoPhaseTrainerGen(
        model=model,
        args=training_args,
        phase1_epochs=cfg.phase1_epochs,
        alpha=cfg.alpha,
        beta=cfg.beta,
        gamma=cfg.gamma,
        gamma_auto=cfg.gamma_auto,
        gamma_target_frac=cfg.gamma_target_frac,
        sigma=cfg.sigma,
        contrast_mode=cfg.contrast_mode,
        contrast_from_start=cfg.contrast_from_start,
        contrastive_dropout=cfg.contrastive_dropout,
        contrastive_tau=cfg.contrastive_tau,
        k_negatives=cfg.k_negatives,
        rl_interval=cfg.rl_interval,
        rl_num_batches=cfg.rl_num_batches,
        rl_subset_size=cfg.rl_subset_size,
        use_reward_baseline=cfg.use_reward_baseline,
        phase2_lr=cfg.phase2_lr,
        divergence_ce_threshold=cfg.divergence_ce_threshold,
        guard_start_step=cfg.guard_start_step,
        rl_dataset=rl_ds,
        rl_references=rl_references,
        tokenizer=tokenizer,
        task_cfg=cfg,
        contrast_neg_in_graph=getattr(cfg, "contrast_neg_in_graph", False),
        reward_baseline_beta=getattr(cfg, "reward_baseline_beta", 0.9),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=causal_lm_collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    return trainer

print("collator + strip_label_tokens + verbalizer_eval + коллбэры (чистый eval) ✓")


collator + strip_label_tokens + verbalizer_eval + коллбэры (чистый eval) ✓


In [8]:
def run_experiment(cfg, experiment_name, init_state=None, return_init_state=False):
    """init_state — общий старт пары; return_init_state=True возвращает
    (scores, snapshot). Сохраняет per-example корректности (для
    парного бутстрепа) и вербализатор-скоринг."""

    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT: {experiment_name}")
    print(f"{'='*70}\n")

    gc.collect()
    torch.cuda.empty_cache()
    set_seed(cfg.seed)

    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    base = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        dtype=dtype,
        trust_remote_code=cfg.trust_remote_code,
    )

    if cfg.gradient_checkpointing:
        base.gradient_checkpointing_enable()
    base.config.use_cache = cfg.use_cache

    peft_cfg = PrefixTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=cfg.num_virtual_tokens,
        prefix_projection=cfg.prefix_projection,
        inference_mode=False,
    )

    model = get_peft_model(base, peft_cfg)
    del base

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.enable_input_require_grads()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    prompt_param_list = [(n, p) for n, p in model.named_parameters()
                         if p.requires_grad and "prompt_encoder" in n]

    start_snapshot = None
    if init_state is not None:
        with torch.no_grad():
            for n, p in prompt_param_list:
                if n in init_state:
                    p.data.copy_(init_state[n].to(p.device))
        print(f"[PairedStart] загружен ОБЩИЙ старт ({len(init_state)} тензоров "
              f"префикса) — probe пропущен\n")
    elif getattr(cfg, "init_selection_inits", 0) > 0:
        select_best_prefix_init(
            model, train_ds, val_ds, tok, cfg,
            n_inits=cfg.init_selection_inits,
            probe_steps=cfg.init_selection_probe_steps,
        )
        if return_init_state:
            start_snapshot = {n: p.data.detach().cpu().clone()
                              for n, p in prompt_param_list}

    print("Trainable parameters:")
    model.print_trainable_parameters()

    trainer = prepare_trainer(
        model, train_ds, val_ds, rl_ds, rl_references, tok, cfg,
        val_references=val_references,
    )

    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    rl_hist = list(getattr(trainer, "rl_reward_history", []) or [])

    print(f"\n{experiment_name} completed in {elapsed/60:.1f} min")

    # Подгружаем ЛУЧШИЙ по val-метрике адаптер
    val_metric_info = {}
    best_path = os.path.join(cfg.output_dir, "best_prefix.pt")
    hist_path = os.path.join(cfg.output_dir, "val_metric_history.json")
    if os.path.exists(best_path):
        best_state = torch.load(best_path, map_location="cpu")
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best_state:
                    p.data.copy_(best_state[n].to(p.device))
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                val_metric_info = json.load(f)
            print(f"Loaded best adapter: val {val_metric_info['metric']}"
                  f"={val_metric_info['best']:.4f} "
                  f"@ epoch {val_metric_info['best_epoch']:.0f} "
                  f"(история: {[round(h['value'], 4) for h in val_metric_info['history']]})")
        else:
            print("Loaded best adapter (best_prefix.pt)")
    else:
        print("best_prefix.pt не найден — оцениваю последнюю эпоху")

    # ---------- ЧИСТЫЙ тест (src-only generate, без золотой метки) ----------
    print(f"\nEvaluating {experiment_name} on test set (clean protocol)...")
    model.eval()

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        collate_fn=causal_lm_collator,
    )

    predictions = []
    references = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask = strip_label_tokens(batch)
            input_ids = input_ids.to(model.device)
            attention_mask = attention_mask.to(model.device)

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=cfg.eval_max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )

            for j in range(input_ids.size(0)):
                pred = tok.decode(
                    gen_ids[j, input_ids.shape[1]:], skip_special_tokens=True
                ).strip()
                predictions.append(pred if pred else " ")

    references = list(test_references)

    per_ex = None
    if cfg.task_kind == "emotion_cls":
        pred_labels = [match_emotion(p) for p in predictions]
        correct_vec = [int(pl == r) for pl, r in zip(pred_labels, references)]
        scores = {"accuracy": float(np.mean(correct_vec))}

        # Вербализатор-скоринг (протокол статьи, детерминированный)
        verb_acc, verb_correct = verbalizer_eval(
            model, test_ds, references, tok)
        scores["verb_accuracy"] = verb_acc
    else:
        rouge = evaluate.load("rouge")
        scores = rouge.compute(
            predictions=predictions,
            references=references,
            rouge_types=["rouge1", "rouge2", "rougeL"],
        )
        # D2: per-example ROUGE-L (для парного бутстрепа) + Distinct-1/2
        per_ex = [float(x) for x in rouge.compute(
            predictions=predictions, references=references,
            rouge_types=["rougeL"], use_aggregator=False)["rougeL"]]
        scores["distinct1"], scores["distinct2"] = distinct_n(predictions)
        correct_vec = None
        verb_correct = None
        pred_labels = None

    print(f"\n{'='*70}")
    print(f"RESULTS: {experiment_name}")
    print(f"{'='*70}")
    for k, v in scores.items():
        print(f"{k}: {v:.4f}")
    print(f"{'='*70}\n")

    if pred_labels is not None:
        miss = [(r, pl, p) for r, pl, p in zip(references, pred_labels, predictions)
                if pl != r][:6]
        print("Примеры ошибок (ref -> pred | генерация):")
        for r, pl, p in miss:
            print(f"  {r:<14} -> {str(pl):<14} | {p[:30]!r}")
    else:
        print("Примеры генераций (test, чистый вход):")
        for r, p in list(zip(references, predictions))[:4]:
            print(f"  REF: {r[:90]!r}")
            print(f"  GEN: {p[:90]!r}")

    results = {
        "experiment": experiment_name,
        "config": {
            "task_kind": cfg.task_kind,
            "model": cfg.model_name,
            "beta": cfg.beta,
            "contrast_mode": getattr(cfg, "contrast_mode", None),
            "contrastive_tau": cfg.contrastive_tau,
            "contrast_from_start": getattr(cfg, "contrast_from_start", False),
            "epochs": cfg.total_epochs,
            "lr": cfg.learning_rate,
            "batch": cfg.batch_size,
            "seed": cfg.seed,
            "paired_common_start": init_state is not None,
            "clean_eval": True,
        },
        "test_scores": scores,
        "correct_vec": correct_vec,
        "verb_correct_vec": verb_correct,
        "per_example_rougeL": per_ex,
        "predictions": predictions,
        "rl_reward_history": rl_hist,
        "val_metric": val_metric_info,
        "training_time_min": elapsed / 60,
    }

    output_file = f"{cfg.output_dir}/results.json"
    os.makedirs(cfg.output_dir, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {output_file}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    if return_init_state:
        return scores, start_snapshot
    return scores


In [10]:
# ============================================================
# СВИП: данные × модели × {MLE, mask, RL}
# ============================================================
import copy
import math

cfg_sweep_base = CFG()          # глобальный cfg уже есть; это база для копий
assert cfg_sweep_base.task_kind == "emotion_cls"

# сетка (зафиксирована до запуска; все ячейки в отчёт):
SWEEP_MODELS = [
    ("Qwen/Qwen2.5-3B",   "3B",  [250, 500, 1000]),   # + 6000 из D1.6 (n=7)
    ("Qwen/Qwen2.5-0.5B", "05B", [250, 1000, 6000]),
]
SWEEP_CONFIGS = ["MLE", "mask", "RL"]
SWEEP_SEEDS = [42, 43, 44]
# NB: токенизатор Qwen2.5 един для всех размеров модели — данные
# переиспользуются без перетокенизации


def rebuild_d1_datasets(train_size):
    """Хвост data-ячейки под новый размер train (val/test/RL фиксированы)."""
    global raw_rl, raw_train, raw_val, raw_test
    global train_ds, val_ds, test_ds, rl_ds
    global rl_references, val_references, test_references
    c = cfg_sweep_base
    assert c.rl_subset_size + train_size <= len(ed_train_full)
    raw_rl    = HFDataset.from_list(ed_train_full[:c.rl_subset_size])
    raw_train = HFDataset.from_list(ed_train_full[c.rl_subset_size:
                                                  c.rl_subset_size + train_size])
    raw_val   = HFDataset.from_list(ed_val_full[:c.val_size])
    raw_test  = HFDataset.from_list(ed_test_full[:c.test_size])
    train_ds = raw_train.map(tokenize_gen, batched=True,
                             remove_columns=raw_train.column_names)
    val_ds   = raw_val.map(tokenize_gen, batched=True,
                           remove_columns=raw_val.column_names)
    test_ds  = raw_test.map(tokenize_gen, batched=True,
                            remove_columns=raw_test.column_names)
    rl_ds    = raw_rl.map(tokenize_gen, batched=True,
                          remove_columns=raw_rl.column_names)
    rl_ds = rl_ds.add_column("ref_idx", list(range(len(rl_ds))))
    rl_references   = raw_rl[c.target_column]
    val_references  = raw_val[c.target_column]
    test_references = raw_test[c.target_column]
    for d in (train_ds, val_ds, test_ds, rl_ds):
        d.set_format("torch")


def make_shared_start(model_name, seed):
    """Случайный (сидированный) старт префикса — общий для всех размеров
    и конфигов данной (модель, сид): парность на двух уровнях."""
    set_seed(seed)
    b = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.bfloat16, trust_remote_code=True)
    m = get_peft_model(b, PrefixTuningConfig(
        task_type=TaskType.CAUSAL_LM, num_virtual_tokens=20,
        prefix_projection=True, inference_mode=False))
    snap = {n: p.data.detach().cpu().clone()
            for n, p in m.named_parameters()
            if p.requires_grad and "prompt_encoder" in n}
    del m, b
    gc.collect(); torch.cuda.empty_cache()
    return snap


os.makedirs("./sweep_ld", exist_ok=True)
_summary_ld = "./sweep_ld/summary_lowdata.json"
ld_rows = []
if os.path.exists(_summary_ld):
    with open(_summary_ld) as f:
        ld_rows = json.load(f).get("rows", [])

def _have(model_short, seed, size):
    return any(r["model"] == model_short and r["seed"] == seed
               and r["size"] == size and all(k in r for k in SWEEP_CONFIGS)
               for r in ld_rows)

for model_name, mshort, sizes in SWEEP_MODELS:
    for seed in SWEEP_SEEDS:
        done_sizes = [n for n in sizes if _have(mshort, seed, n)]
        todo_sizes = [n for n in sizes if n not in done_sizes]
        if not todo_sizes:
            print(f"[Resume] {mshort} s{seed}: все размеры готовы")
            continue
        shared = make_shared_start(model_name, seed)
        for size in todo_sizes:
            rebuild_d1_datasets(size)
            steps = math.ceil(size / cfg_sweep_base.batch_size) * \
                cfg_sweep_base.total_epochs
            row = {"model": mshort, "seed": seed, "size": size}
            for cname in SWEEP_CONFIGS:
                c = copy.copy(cfg_sweep_base)
                c.model_name = model_name
                c.seed = seed
                c.train_size = size
                c.init_selection_inits = 0
                c.warmup_steps = min(200, max(10, steps // 10))
                c.output_dir = f"./sweep_ld/{mshort}_{cname}_s{seed}_n{size}"
                if cname == "MLE":
                    c.beta = 0.0; c.gamma = 0.0; c.gamma_auto = False
                elif cname == "mask":
                    c.beta = 0.1; c.contrast_mode = "mask"
                    c.gamma = 0.0; c.gamma_auto = False
                else:  # RL
                    c.beta = 0.0; c.gamma = 2e-5; c.gamma_auto = True
                rj = os.path.join(c.output_dir, "results.json")
                if os.path.exists(rj):
                    with open(rj) as f:
                        sc = json.load(f)["test_scores"]
                else:
                    sc = run_experiment(
                        c, f"{cname} {mshort} n={size} (s{seed})",
                        init_state=shared)
                row[cname] = sc["accuracy"]
                row[f"{cname}-verb"] = sc["verb_accuracy"]
                gc.collect(); torch.cuda.empty_cache()
            ld_rows = [r for r in ld_rows
                       if not (r["model"] == mshort and r["seed"] == seed
                               and r["size"] == size)] + [row]
            with open(_summary_ld, "w") as f:
                json.dump({"rows": ld_rows}, f, indent=2)
            print(f">>> {mshort} s{seed} n={size}: "
                  f"{ {k: round(row[k], 4) for k in SWEEP_CONFIGS} }\n")
        del shared
        gc.collect(); torch.cuda.empty_cache()

print("Свип завершён:", len(ld_rows), "ячеек")

[Resume] 3B s42: все размеры готовы
[Resume] 3B s43: все размеры готовы
[Resume] 3B s44: все размеры готовы
[Resume] 05B s42: все размеры готовы
[Resume] 05B s43: все размеры готовы
[Resume] 05B s44: все размеры готовы
Свип завершён: 32 ячеек


In [9]:
# ============================================================
# АГРЕГАЦИЯ: кривые data-efficiency + пары к MLE на каждом размере
# (+ точка 3B n=6000 из готового D1.6, n=7 сидов)
# ============================================================
from math import sqrt

d16 = json.load(open("./gen_dialogue_ed_full/summary_d1_6.json"))["per_seed"]
ref6000 = {}
for cname, key in [("MLE", "MLE-only"), ("mask", "contrast-mask"),
                   ("RL", "RL-only")]:
    ref6000[cname] = [r[key] for r in d16]

for model_name, mshort, sizes in SWEEP_MODELS:
    print("\n" + "=" * 88)
    print(f"Кривая data-efficiency: {mshort} (генеративная accuracy, "
          f"mean±std по {len(SWEEP_SEEDS)} сидам)")
    print("=" * 88)
    print(f"{'size':<8}" + "".join(f"{c + ':':<16}" for c in SWEEP_CONFIGS)
          + f"{'mask−MLE':<12}{'RL−MLE':<12}")
    for size in sizes:
        rows_n = [r for r in ld_rows
                  if r["model"] == mshort and r["size"] == size]
        if not rows_n:
            continue
        means = {c: np.mean([r[c] for r in rows_n]) for c in SWEEP_CONFIGS}
        stds = {c: np.std([r[c] for r in rows_n]) for c in SWEEP_CONFIGS}
        dm = [r["mask"] - r["MLE"] for r in rows_n]
        dr = [r["RL"] - r["MLE"] for r in rows_n]
        sem = sqrt(len(dm))
        tm = np.mean(dm) / (np.std(dm, ddof=1) / sqrt(len(dm))) if len(dm) > 1 else float("nan")
        tr = np.mean(dr) / (np.std(dr, ddof=1) / sqrt(len(dr))) if len(dr) > 1 else float("nan")
        print(f"{size:<8}" + "".join(f"{means[c]:.3f}±{stds[c]:.3f}  "
              for c in SWEEP_CONFIGS)
              + f"{np.mean(dm):+.4f}(t={tm:+.1f}) {np.mean(dr):+.4f}(t={tr:+.1f})")
    if mshort == "3B":
        m6 = {c: (np.mean(ref6000[c]), np.std(ref6000[c])) for c in ref6000}
        print(f"6000*   " + "".join(f"{m6[c][0]:.3f}±{m6[c][1]:.3f}  "
              for c in SWEEP_CONFIGS)
              + "(*из D1.6, n=7 сидов)")
print("\nВопрос: растёт ли mask−MLE / RL−MLE при уменьшении данных?")
print("Гипотеза (а приори): вклад CRL максимален в low-resource зоне.")

with open(_summary_ld, "w") as f:
    json.dump({"rows": ld_rows}, f, indent=2)
print("Итог:", _summary_ld)


Кривая data-efficiency: 3B (генеративная accuracy, mean±std по 3 сидам)
size    MLE:            mask:           RL:             mask−MLE    RL−MLE      
250     0.543±0.005  0.537±0.007  0.549±0.010  -0.0053(t=-0.8) +0.0060(t=+0.6)
500     0.567±0.008  0.573±0.018  0.579±0.009  +0.0060(t=+0.4) +0.0120(t=+1.0)
1000    0.586±0.013  0.583±0.012  0.577±0.007  -0.0027(t=-0.2) -0.0093(t=-1.6)
6000*   0.618±0.010  0.628±0.008  0.626±0.006  (*из D1.6, n=7 сидов)

Кривая data-efficiency: 05B (генеративная accuracy, mean±std по 3 сидам)
size    MLE:            mask:           RL:             mask−MLE    RL−MLE      
250     0.377±0.020  0.390±0.016  0.395±0.008  +0.0127(t=+2.6) +0.0173(t=+1.3)
1000    0.513±0.018  0.501±0.032  0.510±0.015  -0.0127(t=-0.6) -0.0033(t=-1.2)
6000    0.541±0.005  0.551±0.012  0.557±0.009  +0.0093(t=+1.1) +0.0160(t=+3.8)

Вопрос: растёт ли mask−MLE / RL−MLE при уменьшении данных?
Гипотеза (а приори): вклад CRL максимален в low-resource зоне.
Итог: ./sweep_ld/summary_

In [10]:
# ============================================================
# РАСШИРЕНИЕ 0.5B-ячеек до n=7 (сиды 45-48), ПРЕДЗАЯВЛЕННО:
# обе ячейки (n=250 и n=6000), все 3 конфига, все сиды в отчёт,
# дальнейших расширений не будет. Мотивация: 0.5B×6000 RL−MLE =
# +0.016, 3/3 сида, каждый дифф ≥ +0.008 (t=3.8 при критe 4.303 —
# не хватило мощности). Время ~80 мин (0.5B: MLE ~3.3, mask ~6.5,
# RL ~5.3 мин).
# ============================================================
import copy
import math

EXT05B = [("05B", "Qwen/Qwen2.5-0.5B", 250), ("05B", "Qwen/Qwen2.5-0.5B", 6000)]
EXT_SEEDS = [45, 46, 47, 48]

for mshort, model_name, size in EXT05B:
    for seed in EXT_SEEDS:
        if any(r["model"] == mshort and r["seed"] == seed and r["size"] == size
               and all(k in r for k in SWEEP_CONFIGS) for r in ld_rows):
            print(f"[Resume] {mshort} s{seed} n={size} готов")
            continue
        shared = make_shared_start(model_name, seed)
        rebuild_d1_datasets(size)
        steps = math.ceil(size / cfg_sweep_base.batch_size) * \
            cfg_sweep_base.total_epochs
        row = {"model": mshort, "seed": seed, "size": size}
        for cname in SWEEP_CONFIGS:
            c = copy.copy(cfg_sweep_base)
            c.model_name = model_name
            c.seed = seed
            c.train_size = size
            c.init_selection_inits = 0
            c.warmup_steps = min(200, max(10, steps // 10))
            c.output_dir = f"./sweep_ld/{mshort}_{cname}_s{seed}_n{size}"
            if cname == "MLE":
                c.beta = 0.0; c.gamma = 0.0; c.gamma_auto = False
            elif cname == "mask":
                c.beta = 0.1; c.contrast_mode = "mask"
                c.gamma = 0.0; c.gamma_auto = False
            else:
                c.beta = 0.0; c.gamma = 2e-5; c.gamma_auto = True
            rj = os.path.join(c.output_dir, "results.json")
            if os.path.exists(rj):
                with open(rj) as f:
                    sc = json.load(f)["test_scores"]
            else:
                sc = run_experiment(
                    c, f"{cname} {mshort} n={size} (s{seed})", init_state=shared)
            row[cname] = sc["accuracy"]
            row[f"{cname}-verb"] = sc["verb_accuracy"]
            gc.collect(); torch.cuda.empty_cache()
        ld_rows = [r for r in ld_rows
                   if not (r["model"] == mshort and r["seed"] == seed
                           and r["size"] == size)] + [row]
        with open(_summary_ld, "w") as f:
            json.dump({"rows": ld_rows}, f, indent=2)
        print(f">>> {mshort} s{seed} n={size}: "
              f"{ {k: round(row[k], 4) for k in SWEEP_CONFIGS} }\n")
        del shared
        gc.collect(); torch.cuda.empty_cache()

# вердикт по расширенным ячейкам
from math import sqrt
for size in (250, 6000):
    rs = [r for r in ld_rows if r["model"] == "05B" and r["size"] == size]
    n = len(rs)
    if n < 4:
        continue
    print(f"\n0.5B n={size} (n_сидов={n}):")
    for a in ("mask", "RL"):
        dd = [r[a] - r["MLE"] for r in rs]
        se = np.std(dd, ddof=1) / sqrt(n)
        t = np.mean(dd) / se if se > 0 else float("nan")
        crit = {4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447}[n]
        print(f"  {a}−MLE: {np.mean(dd):+.4f} ± {np.std(dd, ddof=1):.4f} | "
              f"t={t:+.2f} (крит {crit}) | per-seed "
              f"{[f'{x:+.4f}' for x in dd]} | "
              f"{'ЗНАЧИМО' if abs(t) > crit else 'н.з.'}")

Map: 100%|██████████| 600/600 [00:00<00:00, 8345.27 examples/s]



STARTING EXPERIMENT: MLE 05B n=250 (s45)



Loading weights: 100%|██████████| 290/290 [00:00<00:00, 8819.42it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[Step 0] CE/token (EMA): 8.689 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,3.837946,3.362440
2,2.484131,2.840578
3,1.948570,2.580890
4,1.739165,2.558181


...

[Step 10] CE/token (EMA): 11.145 | Contrast[mask]: 0.690 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 10.271 | Contrast[mask]: 0.687 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 9.120 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 7.706 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 6.438 | Contrast[mask]: 0.682 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 5.491 | Contrast[mask]: 0.687 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 4.906 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 4.530 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 3.990 | Contrast[mask]: 0.684 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 3.716 | Contrast[mask]: 0.688 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 3.435 | Contrast[mask]: 0.683 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 3.166 | Contrast[mask]: 0.685 | RL

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9625.52it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 2.205e-04
[RL sample] REF: 'prepared'
[RL sample] GEN: 'Happiness\nHypnaca'
[RL step 0] reward=0.0078 adv=+0.0078 gamma=2.20e-04
[Step 0] CE/token (EMA): 11.617 | Contrast[mask]: 0.000 | RL: -3.485 | Reward: 0.0078


Epoch,Training Loss,Validation Loss
1,1.465585,-0.282763
2,1.259746,-1.905035
3,1.320469,-0.147679
4,1.270414,3.234268


[Step 10] CE/token (EMA): 11.150 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 10.271 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 9.118 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 7.707 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'content'
[RL sample] GEN: 'fat.\n$\\end1$:'
[RL step 50] reward=0.2188 adv=+0.2180 gamma=2.20e-04
[Step 50] CE/token (EMA): 6.425 | Contrast[mask]: 0.000 | RL: -97.259 | Reward: 0.2188
[Step 60] CE/token (EMA): 5.483 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 4.895 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 4.511 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 3.978 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'disappointed'
[RL sample] GEN: 'sad.\nBased on the text'
[RL step 100] reward=0.4062 adv=+0.3837 

In [10]:
# ============================================================
# КОНТРОЛЬНЫЕ CAPACITY-ТОЧКИ (критика-3 §22): 1.5B и 3B при n=6000
# ПО ЕДИНОМУ sweep-ПРОТОКОЛЮ (без probe, общий случайный старт на
# (модель,сид)) — уравнивает init-протокол с 0.5B и даёт трёхточечную
# dose-response кривую: 0.5B (n=7) -> 1.5B (n=3) -> 3B (n=3).
# Отдельная сверка: 3B probe-протокол из D1.6 (RL−MLE +0.0043, n=10).
# Время: 3B ~37 мин/сид, 1.5B ~22 мин/сид -> ~3 ч суммарно. Resume-safe.
# ============================================================
import copy
import math

CAP_POINTS = [("15B", "Qwen/Qwen2.5-1.5B"), ("3B", "Qwen/Qwen2.5-3B")]
CAP_SIZE, CAP_SEEDS = 6000, [42, 43, 44]

for mshort, model_name in CAP_POINTS:
    for seed in CAP_SEEDS:
        if any(r["model"] == mshort and r["seed"] == seed and r["size"] == CAP_SIZE
               and all(k in r for k in SWEEP_CONFIGS) for r in ld_rows):
            print(f"[Resume] {mshort} s{seed} n={CAP_SIZE} готов")
            continue
        shared = make_shared_start(model_name, seed)
        rebuild_d1_datasets(CAP_SIZE)
        steps = math.ceil(CAP_SIZE / cfg_sweep_base.batch_size) * \
            cfg_sweep_base.total_epochs
        row = {"model": mshort, "seed": seed, "size": CAP_SIZE}
        for cname in SWEEP_CONFIGS:
            c = copy.copy(cfg_sweep_base)
            c.model_name = model_name
            c.seed = seed
            c.train_size = CAP_SIZE
            c.init_selection_inits = 0
            c.warmup_steps = min(200, max(10, steps // 10))
            c.output_dir = f"./sweep_ld/{mshort}_{cname}_s{seed}_n{CAP_SIZE}"
            if cname == "MLE":
                c.beta = 0.0; c.gamma = 0.0; c.gamma_auto = False
            elif cname == "mask":
                c.beta = 0.1; c.contrast_mode = "mask"
                c.gamma = 0.0; c.gamma_auto = False
            else:
                c.beta = 0.0; c.gamma = 2e-5; c.gamma_auto = True
            rj = os.path.join(c.output_dir, "results.json")
            if os.path.exists(rj):
                with open(rj) as f:
                    sc = json.load(f)["test_scores"]
            else:
                sc = run_experiment(c, f"{cname} {mshort} n={CAP_SIZE} (s{seed})",
                                    init_state=shared)
            row[cname] = sc["accuracy"]
            row[f"{cname}-verb"] = sc["verb_accuracy"]
            gc.collect(); torch.cuda.empty_cache()
        ld_rows = [r for r in ld_rows
                   if not (r["model"] == mshort and r["seed"] == seed
                           and r["size"] == CAP_SIZE)] + [row]
        with open(_summary_ld, "w") as f:
            json.dump({"rows": ld_rows}, f, indent=2)
        print(f">>> {mshort} s{seed} n={CAP_SIZE}: "
              f"{ {k: round(row[k], 4) for k in SWEEP_CONFIGS} }\n")
        del shared
        gc.collect(); torch.cuda.empty_cache()

# ---------- вердикт: dose-response по ёмкости ----------
from math import sqrt

tc = {3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447}
print("\n" + "=" * 92)
print("DOSE-RESPONSE ПО ЁМКОСТИ (единый sweep-протокол: без probe,")
print("общий случайный старт на (модель,сид), n=6000)")
print("=" * 92)
print(f"{'модель':<8}{'MLE':<10}{'mask−MLE':<28}{'RL−MLE':<28}")
for mshort in ("05B", "15B", "3B"):
    rs = [r for r in ld_rows if r["model"] == mshort and r["size"] == 6000]
    if not rs:
        continue
    line = f"{mshort:<8}{np.mean([r['MLE'] for r in rs]):<10.3f}"
    for a in ("mask", "RL"):
        dd = [r[a] - r["MLE"] for r in rs]
        n = len(dd)
        se = np.std(dd, ddof=1) / sqrt(n)
        t = np.mean(dd) / se if se > 0 else float("nan")
        sig = abs(t) > tc.get(n, 2.447)
        line += (f"{np.mean(dd):+.4f} (t={t:+.1f}"
                 f"{' ЗНАЧ' if sig else ''}, n={n})   ")
    print(line)
print("\nСверка с probe-протоколом (D1.6, 3B): RL−MLE +0.0043 (n=10, 8/10,")
print("бутстреп p=0.023); mask −0.0002 (n=7, большой тест). Если sweep-3B")
print("воспроизводит порядок эффектов — capacity-заявка протокольно чиста.")
print("\nГипотеза (предзаявлена, ROADMAP A): эффект RL убывает с ростом")
print("ёмкости модели; кривая 0.5B > 1.5B > 3B monotone по mean-разности.")

Map: 100%|██████████| 600/600 [00:00<00:00, 8223.67 examples/s]



STARTING EXPERIMENT: MLE 15B n=6000 (s42)



Loading weights: 100%|██████████| 338/338 [00:00<00:00, 9058.34it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 3,755,264 || all params: 1,547,469,568 || trainable%: 0.2427
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[Step 0] CE/token (EMA): 5.905 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,1.297028,1.526584
2,1.307822,1.204804
3,0.881052,1.107846
4,0.982979,1.120036


...

[Step 10] CE/token (EMA): 4.432 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 4.376 | Contrast[mask]: 0.691 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 4.211 | Contrast[mask]: 0.688 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 3.964 | Contrast[mask]: 0.689 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 3.501 | Contrast[mask]: 0.682 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 3.299 | Contrast[mask]: 0.688 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 3.212 | Contrast[mask]: 0.682 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.988 | Contrast[mask]: 0.690 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.805 | Contrast[mask]: 0.685 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 2.655 | Contrast[mask]: 0.685 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 2.570 | Contrast[mask]: 0.690 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 2.270 | Contrast[mask]: 0.685 | RL: 

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9480.13it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 5.002e-07
[RL sample] REF: 'prepared'
[RL sample] GEN: "anxious\nI'm sorry to"
[RL step 0] reward=0.1953 adv=+0.1953 gamma=5.00e-07
[Step 0] CE/token (EMA): 3.903 | Contrast[mask]: 0.000 | RL: -1.171 | Reward: 0.1953


Epoch,Training Loss,Validation Loss
1,1.051621,1.279484
2,1.083961,1.121169
3,1.113773,1.038484
4,0.942583,1.092260


[Step 10] CE/token (EMA): 4.432 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 4.379 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 4.213 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 3.967 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'content'
[RL sample] GEN: 'happy\nRelationship: friend'
[RL step 50] reward=0.3281 adv=+0.3086 gamma=5.00e-07
[Step 50] CE/token (EMA): 3.507 | Contrast[mask]: 0.000 | RL: -1.850 | Reward: 0.3281
[Step 60] CE/token (EMA): 3.308 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 3.207 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 3.018 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.848 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'disappointed'
[RL sample] GEN: 'disappointed, sad\n\nWhat is'
[RL step 100] reward=0.515

In [11]:
# ============================================================
# НЕЗАВИСИМАЯ РЕПЛИКАЦИЯ 0.5B×6000, сиды 52–54 (критика-4 §17)
# Протокол заморожен заранее (§8.1/§10 журнала): те же
# гиперпараметры, общий случайный старт на сид, {MLE, mask, CE+RL}.
# Все сиды включаются без отбора. ~45 мин (0.5B).
# ============================================================
import copy
import math

REPL_SEEDS = [52, 53, 54]
REPL_SIZE = 6000

for seed in REPL_SEEDS:
    if any(r["model"] == "05B" and r["seed"] == seed and r["size"] == REPL_SIZE
           and all(k in r for k in SWEEP_CONFIGS) for r in ld_rows):
        print(f"[Resume] 05B s{seed} готов")
        continue
    shared = make_shared_start("Qwen/Qwen2.5-0.5B", seed)
    rebuild_d1_datasets(REPL_SIZE)
    steps = math.ceil(REPL_SIZE / cfg_sweep_base.batch_size) * \
        cfg_sweep_base.total_epochs
    row = {"model": "05B", "seed": seed, "size": REPL_SIZE, "replication": True}
    for cname in SWEEP_CONFIGS:
        c = copy.copy(cfg_sweep_base)
        c.model_name = "Qwen/Qwen2.5-0.5B"
        c.seed = seed
        c.train_size = REPL_SIZE
        c.init_selection_inits = 0
        c.warmup_steps = min(200, max(10, steps // 10))
        c.output_dir = f"./sweep_ld/05B_{cname}_s{seed}_n{REPL_SIZE}"
        if cname == "MLE":
            c.beta = 0.0; c.gamma = 0.0; c.gamma_auto = False
        elif cname == "mask":
            c.beta = 0.1; c.contrast_mode = "mask"
            c.gamma = 0.0; c.gamma_auto = False
        else:
            c.beta = 0.0; c.gamma = 2e-5; c.gamma_auto = True
        rj = os.path.join(c.output_dir, "results.json")
        if os.path.exists(rj):
            with open(rj) as f:
                sc = json.load(f)["test_scores"]
        else:
            sc = run_experiment(c, f"{cname} 05B repl n={REPL_SIZE} (s{seed})",
                                init_state=shared)
        row[cname] = sc["accuracy"]
        row[f"{cname}-verb"] = sc["verb_accuracy"]
        gc.collect(); torch.cuda.empty_cache()
    ld_rows = [r for r in ld_rows
               if not (r["model"] == "05B" and r["seed"] == seed
                       and r["size"] == REPL_SIZE)] + [row]
    with open(_summary_ld, "w") as f:
        json.dump({"rows": ld_rows}, f, indent=2)
    print(f">>> repl 05B s{seed}: "
          f"{ {k: round(row[k], 4) for k in SWEEP_CONFIGS} }\n")
    del shared
    gc.collect(); torch.cuda.empty_cache()

# ---------- агрегация: n=10 (исходные 42-48 + репликация 52-54) ----------
from math import sqrt, comb

rs = [r for r in ld_rows if r["model"] == "05B" and r["size"] == 6000]
rs = sorted(rs, key=lambda r: r["seed"])
print("\n" + "=" * 90)
print(f"ГЛАВНЫЙ РЕЗУЛЬТАТ, РЕПЛИКАЦИЯ: 0.5B x 6000, n={len(rs)} сидов "
      f"(42-48 исходные + 52-54 репликация)")
print("=" * 90)
for a in ("RL", "mask"):
    d = [r[a] - r["MLE"] for r in rs]
    n = len(d); se = np.std(d, ddof=1) / sqrt(n)
    t = np.mean(d) / se
    crit = {10: 2.262, 9: 2.262, 8: 2.306, 7: 2.447}[n]
    k = sum(1 for x in d if x > 0)
    psign = 2 * sum(comb(n, i) for i in range(k, n + 1)) / 2**n
    orig = [x for x, r in zip(d, rs) if r["seed"] <= 48]
    repl = [x for x, r in zip(d, rs) if r["seed"] >= 52]
    print(f"{a}-MLE: {np.mean(d):+.4f} ± {np.std(d, ddof=1):.4f} | "
          f"t={t:+.2f} (крит {crit}) | {k}/{n} в плюс, знак. p={psign:.4f}")
    print(f"   исходные (n={len(orig)}): mean {np.mean(orig):+.4f} | "
          f"репликация (n={len(repl)}): mean "
          f"{np.mean(repl):+.4f} {[f'{x:+.4f}' for x in repl]}")
print("\nКритерий согласованности: среднее репликации внутри CI исходных?")
print("Гипотеза (предзаявлена): направление и порядок величины сохраняются.")

Map: 100%|██████████| 600/600 [00:00<00:00, 8702.15 examples/s]



STARTING EXPERIMENT: MLE 05B repl n=6000 (s52)



Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9746.30it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[Step 0] CE/token (EMA): 10.018 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,1.718141,1.587759
2,1.702261,1.443275
3,1.394044,1.293718
4,1.206875,1.271789


...

[Step 10] CE/token (EMA): 10.534 | Contrast[mask]: 0.690 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 10.128 | Contrast[mask]: 0.691 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 9.313 | Contrast[mask]: 0.691 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 7.906 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 6.748 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 5.677 | Contrast[mask]: 0.684 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 5.109 | Contrast[mask]: 0.683 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 4.383 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 3.940 | Contrast[mask]: 0.684 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 3.543 | Contrast[mask]: 0.687 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 3.263 | Contrast[mask]: 0.678 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 2.960 | Contrast[mask]: 0.686 | RL

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9984.80it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 9.748e-05
[RL sample] REF: 'prepared'
[RL sample] GEN: '1.64'
[RL step 0] reward=0.0156 adv=+0.0156 gamma=9.75e-05
[Step 0] CE/token (EMA): 10.274 | Contrast[mask]: 0.000 | RL: -3.082 | Reward: 0.0156


Epoch,Training Loss,Validation Loss
1,1.654452,0.636113
2,1.652834,0.238890
3,1.260151,0.535742
4,1.027834,2.193464


[Step 10] CE/token (EMA): 10.531 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 10.131 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 9.315 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 7.899 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'content'
[RL sample] GEN: 'fat.\nCiting and not'
[RL step 50] reward=0.2109 adv=+0.2094 gamma=9.75e-05
[Step 50] CE/token (EMA): 6.735 | Contrast[mask]: 0.000 | RL: -41.310 | Reward: 0.2109
[Step 60] CE/token (EMA): 5.666 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 5.092 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 4.377 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 3.929 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'disappointed'
[RL sample] GEN: 'sad.\nVerb: grieve'
[RL step 100] reward=0.3984 adv=+0.3759 

In [12]:
# ============================================================
# PREFIX-DRIFT ДИАГНОСТИКА: ||P_best - P_0|| / ||P_0|| (критика-4 §35)
# Вопрос: «вспомогательная цель не даёт сигнала» или «даёт чрезмерный
# дрейф промпта»? Считаем относительное смещение обученного промпта
# от стартового. P_0: shared_init.pt (3B-серии) или воспроизведённый
# случайный старт (0.5B-sweep). CPU-часть (3B) выполняется без
# загрузки моделей; 0.5B требует коротких загрузок (~15 с x сид).
# ============================================================
import glob

def _drift(P0, Pb):
    num = sum(float((Pb[k].float() - P0[k].float()).norm() ** 2)
              for k in P0 if k in Pb) ** 0.5
    den = sum(float(P0[k].float().norm() ** 2) for k in P0) ** 0.5
    return num / max(den, 1e-12)

DEGRAD = {("gen_d2_incoherent_d21_seed42",): "incoherent s42 (0.128)",
          ("gen_d2_mask_d21_seed48",): "mask s48 (0.136)",
          ("gen_dialogue_ed_contrast_wl_d16_seed48",): "wl s48 (0.300)"}

def group_drift(name, run_dirs, start_of):
    vals, notes = [], []
    for d in run_dirs:
        bp = os.path.join(d, "best_prefix.pt")
        P0p = start_of(d)
        if not (os.path.exists(bp) and P0p and os.path.exists(P0p)):
            continue
        P0, Pb = torch.load(P0p, map_location="cpu"), torch.load(bp, map_location="cpu")
        vals.append(_drift(P0, Pb))
        tag = DEGRAD.get((d.replace("./", ""),), DEGRAD.get((d,), ""))
        if tag:
            notes.append(f"  ! {os.path.basename(d)}: {tag} -> drift {vals[-1]:.3f}")
    if vals:
        print(f"{name:<42} {np.mean(vals):.3f} ± {np.std(vals):.3f} "
              f"(n={len(vals)})")
        for nt in notes:
            print(nt)
    return vals

results = {}

# --- D2 (3B): старт из mle-папки того же сида ---
for cfg, bdir, wave, seeds in [
        ("D2 MLE", "gen_d2_mle", "d21", range(42, 49)),
        ("D2 маска", "gen_d2_mask", "d21", range(42, 49)),
        ("D2 неуместный", "gen_d2_offtopic", "d21", range(42, 49)),
        ("D2 переставленный", "gen_d2_incoherent", "d21", range(42, 49)),
        ("D2 CE+RL", "gen_d2_rl_only", "d22", range(42, 49)),
        ("D2 GRPO", "gen_d2_grpo", "d24", (42, 43, 44))]:
    dirs = [f"{bdir}_{wave}_seed{s}" for s in seeds]
    results[cfg] = group_drift(
        cfg, dirs,
        lambda d: os.path.join(f"gen_d2_mle_d21_seed{d.split('seed')[-1]}",
                               "shared_init.pt"))

# --- D1 (3B): старт из mle-папки того же сида ---
for cfg, bdir, seeds in [
        ("D1 MLE", "gen_crl_mle_baseline_d16", range(42, 49)),
        ("D1 маска", "gen_dialogue_ed_contrast_mask_d16", range(42, 49)),
        ("D1 путаемая метка", "gen_dialogue_ed_contrast_wl_d16", range(42, 49)),
        ("D1 CE+RL", "gen_dialogue_ed_rl_only_d16", range(42, 49)),
        ("D1 полный CRL", "gen_dialogue_ed_full_crl_d16", range(42, 49))]:
    dirs = [f"{bdir}_seed{s}" for s in seeds]
    results[cfg] = group_drift(
        cfg, dirs,
        lambda d: os.path.join(f"gen_crl_mle_baseline_d16_seed{d.split('seed')[-1]}",
                               "shared_init.pt"))

# --- 0.5B×6000 (sweep): старт воспроизводится make_shared_start ---
zs_dirs = {"MLE": "MLE", "mask": "mask", "RL": "RL"}
for label, key in zs_dirs.items():
    vals = []
    for seed in sorted({r["seed"] for r in ld_rows
                        if r["model"] == "05B" and r["size"] == 6000}):
        d = f"sweep_ld/05B_{key}_s{seed}_n6000"
        bp = os.path.join(d, "best_prefix.pt")
        if not os.path.exists(bp):
            continue
        P0 = make_shared_start("Qwen/Qwen2.5-0.5B", seed)
        Pb = torch.load(bp, map_location="cpu")
        vals.append(_drift(P0, Pb))
        del P0
    if vals:
        results[f"0.5B {label}"] = vals
        print(f"{'0.5B x6000 ' + label:<42} {np.mean(vals):.3f} ± {np.std(vals):.3f} "
              f"(n={len(vals)})")

json.dump({k: [float(x) for x in v] for k, v in results.items()},
          open("sweep_ld/prefix_drift.json", "w"), indent=2)
print("\nСохранено: sweep_ld/prefix_drift.json")
print("\nИнтерпретация: деградации <-> большой дрейф? 0.5B: RL > MLE")
print("(цель реально двигает промпт)? D2 3B: одинаковый дрейф ->")
print("«нет сигнала», а не «нет движения».")

D2 MLE                                     0.275 ± 0.045 (n=7)
D2 маска                                   0.266 ± 0.047 (n=7)
  ! gen_d2_mask_d21_seed48: mask s48 (0.136) -> drift 0.296
D2 неуместный                              0.267 ± 0.051 (n=7)
D2 переставленный                          0.278 ± 0.040 (n=7)
  ! gen_d2_incoherent_d21_seed42: incoherent s42 (0.128) -> drift 0.228
D2 CE+RL                                   0.260 ± 0.042 (n=7)
D2 GRPO                                    0.281 ± 0.045 (n=3)
D1 MLE                                     0.292 ± 0.011 (n=7)
D1 маска                                   0.294 ± 0.016 (n=7)
D1 путаемая метка                          0.268 ± 0.039 (n=7)
  ! gen_dialogue_ed_contrast_wl_d16_seed48: wl s48 (0.300) -> drift 0.237
D1 CE+RL                                   0.295 ± 0.010 (n=7)
D1 полный CRL                              0.300 ± 0.010 (n=7)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9277.88it/s]


0.5B x6000 MLE                             0.234 ± 0.006 (n=10)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9994.23it/s]


0.5B x6000 mask                            0.231 ± 0.008 (n=10)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9767.98it/s]


0.5B x6000 RL                              0.231 ± 0.006 (n=10)

Сохранено: sweep_ld/prefix_drift.json

Интерпретация: деградации <-> большой дрейф? 0.5B: RL > MLE
(цель реально двигает промпт)? D2 3B: одинаковый дрейф ->
«нет сигнала», а не «нет движения».


In [14]:
# ============================================================
# РЕПЛИКАЦИЯ mask x 250 (сиды 52–54) — последний непроверенный
# разведочный результат (§10.3). Протокол sw_ext05b без изменений:
# общий случайный старт на сид, все 3 конфигурации (для полноты),
# все сиды в отчёт. 250 примеров -> самые быстрые прогоны (~1 мин),
# итого ~15 мин.
# ============================================================
import copy
import math
from math import sqrt, comb

REPL250_SIZE = 250

# tokenize_gen читает ГЛОБАЛЬНЫЙ cfg (ячейка данных замыкается на модуль
# ноутбука). В прошлой сессии в ядре глобальный cfg оказался затёрт
# строкой — отсюда AttributeError: 'str' object has no attribute
# 'max_total_len' на первом же map(). Восстанавливаем базовый конфиг
# свипа: значения идентичны CFG(), на числовой путь не влияет.
cfg = cfg_sweep_base

for seed in (52, 53, 54):
    if any(r["model"] == "05B" and r["seed"] == seed and r["size"] == REPL250_SIZE
           and all(k in r for k in SWEEP_CONFIGS) for r in ld_rows):
        print(f"[Resume] 05B s{seed} n=250 готов")
        continue
    shared = make_shared_start("Qwen/Qwen2.5-0.5B", seed)
    rebuild_d1_datasets(REPL250_SIZE)
    steps = math.ceil(REPL250_SIZE / cfg_sweep_base.batch_size) * \
        cfg_sweep_base.total_epochs
    row = {"model": "05B", "seed": seed, "size": REPL250_SIZE, "replication": True}
    for cname in SWEEP_CONFIGS:
        c = copy.copy(cfg_sweep_base)
        c.model_name = "Qwen/Qwen2.5-0.5B"
        c.seed = seed
        c.train_size = REPL250_SIZE
        c.init_selection_inits = 0
        c.warmup_steps = min(200, max(10, steps // 10))
        c.output_dir = f"./sweep_ld/05B_{cname}_s{seed}_n{REPL250_SIZE}"
        if cname == "MLE":
            c.beta = 0.0; c.gamma = 0.0; c.gamma_auto = False
        elif cname == "mask":
            c.beta = 0.1; c.contrast_mode = "mask"
            c.gamma = 0.0; c.gamma_auto = False
        else:
            c.beta = 0.0; c.gamma = 2e-5; c.gamma_auto = True
        rj = os.path.join(c.output_dir, "results.json")
        if os.path.exists(rj):
            with open(rj) as f:
                sc = json.load(f)["test_scores"]
        else:
            sc = run_experiment(c, f"{cname} 05B repl n={REPL250_SIZE} (s{seed})",
                                init_state=shared)
        row[cname] = sc["accuracy"]
        row[f"{cname}-verb"] = sc["verb_accuracy"]
        gc.collect(); torch.cuda.empty_cache()
    ld_rows = [r for r in ld_rows
               if not (r["model"] == "05B" and r["seed"] == seed
                       and r["size"] == REPL250_SIZE)] + [row]
    with open(_summary_ld, "w") as f:
        json.dump({"rows": ld_rows}, f, indent=2)
    print(f">>> repl 05B s{seed} n=250: "
          f"{ {k: round(row[k], 4) for k in SWEEP_CONFIGS} }\n")
    del shared
    gc.collect(); torch.cuda.empty_cache()

# ---------- агрегация n=10 ----------
rs = sorted([r for r in ld_rows if r["model"] == "05B" and r["size"] == 250],
            key=lambda r: r["seed"])
print("\n" + "=" * 88)
print(f"РЕПЛИКАЦИЯ mask x 250: n={len(rs)} сидов (42-48 исходные + 52-54)")
print("=" * 88)
for a in ("mask", "RL"):
    d = [r[a] - r["MLE"] for r in rs]
    n = len(d)
    se = np.std(d, ddof=1) / sqrt(n)
    t = np.mean(d) / se if se > 0 else float("nan")
    crit = {7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262}.get(n, 2.262)  # t.975, df=n-1
    k = sum(1 for x in d if x > 0)
    nz = sum(1 for x in d if abs(x) > 1e-12)
    psign = 2 * sum(comb(nz, i) for i in range(k, nz + 1)) / 2**nz if nz else 1.0
    orig = [x for x, r in zip(d, rs) if r["seed"] <= 48]
    repl = [x for x, r in zip(d, rs) if r["seed"] >= 52]
    print(f"{a}-MLE: {np.mean(d):+.4f} ± {np.std(d, ddof=1):.4f} | "
          f"t={t:+.2f} (крит {crit}) | {k}/{n} в плюс, p_sign={psign:.4f}")
    print(f"   исходные (n={len(orig)}): {np.mean(orig):+.4f} | "
          f"репликация (n={len(repl)}): {np.mean(repl):+.4f} "
          f"{[f'{x:+.4f}' for x in repl]}")
print("\nИсход (s52-54 x 6000): CE+RL репликация была -0.006 — эффект")
print("не воспроизвёлся. Здесь последний непроверенный результат работы.")

Map: 100%|██████████| 600/600 [00:00<00:00, 8308.49 examples/s]



STARTING EXPERIMENT: MLE 05B repl n=250 (s52)



Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9394.46it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[Step 0] CE/token (EMA): 11.382 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,4.012821,3.637493
2,2.265661,2.787102
3,2.196545,2.550430
4,1.984441,2.506881


...

[Step 10] CE/token (EMA): 10.005 | Contrast[mask]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 7.847 | Contrast[mask]: 0.689 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 6.129 | Contrast[mask]: 0.688 | RL: 0.000 | Reward: 0.0000
[ValMetric epoch 1] accuracy=0.2850 | best=0.2850 @ epoch 1
[Step 40] CE/token (EMA): 3.740 | Contrast[mask]: 0.688 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 3.425 | Contrast[mask]: 0.682 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 3.135 | Contrast[mask]: 0.688 | RL: 0.000 | Reward: 0.0000
[Canary epoch 2] terrified
contemplate
poss | terrified
description: terrified | excited

desires: anxious | terrified
description: terrified
[ValMetric epoch 2] accuracy=0.3250 | best=0.3250 @ epoch 2
[Step 70] CE/token (EMA): 2.637 | Contrast[mask]: 0.685 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.479 | Contrast[mask]: 0.683 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.318 | Contrast[mask]: 0.684 |

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9489.01it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 1.024e-04
[RL sample] REF: 'prepared'
[RL sample] GEN: '1.64'
[RL step 0] reward=0.0156 adv=+0.0156 gamma=1.02e-04
[Step 0] CE/token (EMA): 10.793 | Contrast[mask]: 0.000 | RL: -3.238 | Reward: 0.0156


Epoch,Training Loss,Validation Loss
1,3.571870,3.488765
2,-5.386384,2.949403
3,2.107378,2.716308
4,1.708070,2.662394


[Step 10] CE/token (EMA): 10.004 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 7.847 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 6.143 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[ValMetric epoch 1] accuracy=0.2800 | best=0.2800 @ epoch 1
[Step 40] CE/token (EMA): 3.654 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'content'
[RL sample] GEN: 'comfortable.\nbelief: I have'
[RL step 50] reward=0.3906 adv=+0.3891 gamma=1.02e-04
[Step 50] CE/token (EMA): 3.382 | Contrast[mask]: 0.000 | RL: -80.641 | Reward: 0.3906
[Step 60] CE/token (EMA): 3.125 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Canary epoch 2] terrified.
Fill in the blank | terrified. 201 | excited.

in the situation, | scared
to gushing
[ValMetric epoch 2] accuracy=0.3150 | best=0.3150 @ epoch 2
[Step 70] CE/token (EMA): 2.645 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.488 | Contra